# 🧠 Medical Image ROI Extraction — Model Training
## EfficientNet-B3 + GradCAM++ | X-Ray · CT Scan · MRI Brain Tumor

### Architecture & Design Decisions

| Component | Choice | Reason |
|---|---|---|
| Backbone | EfficientNet-B3 | Best accuracy/params trade-off for medical imaging |
| Explainability | GradCAM++ | Better localisation than GradCAM for small lesions |
| Head | Dropout(0.5) → FC(256) → ReLU → Dropout(0.3) → FC(n_classes) | Aggressive regularisation |
| Loss | CrossEntropyLoss(label_smoothing=0.15) | Prevents overconfident predictions |
| Augmentation | MixUp(α=0.3) + Flip + Rotation + ColorJitter | Generalisation without distorting anatomy |
| Optimizer | AdamW(weight_decay=2e-4) | Weight decay prevents overfitting |
| LR Schedule | LR warmup (3 ep) + CosineAnnealing | Stable convergence |
| Early Stopping | val_loss patience=7 | Stops before overfitting |
| Unfreezing | Selective blocks only | Avoids catastrophic forgetting of ImageNet features |

### Anti-Overfitting Measures (fixes >95% accuracy issue)
- Label smoothing 0.15 → model never learns to be 100% confident
- MixUp α=0.3 → forces learning of intermediate representations  
- Dropout 0.5/0.3 → two stages of regularisation in the head
- Early stopping on val_loss → stops exactly when generalisation peaks
- Weight decay 2e-4 → L2 penalty on all weights

### CPU/GPU Compatibility
- AMP (mixed precision) only on CUDA — CPU skips it cleanly
- GradScaler only on CUDA — no crash on CPU
- Adaptive batch size: 32 (GPU) / 16 (CPU)
- num_workers=0 always — safe on Windows and CPU-only systems

In [1]:
# ── CELL 1 — Imports & Config ────────────────────────────────────────────────
import os, gc, sys, time, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive — works everywhere including CPU servers
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import cv2
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_fscore_support
from PIL import Image as PILImage
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device — auto-detect, works on both CPU and GPU ──────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP    = (DEVICE.type == 'cuda')   # AMP only on CUDA
PIN_MEMORY = (DEVICE.type == 'cuda')

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path('preprocessed')   # output from preprocessing.ipynb
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32 if DEVICE.type == 'cuda' else 16  # smaller on CPU → less RAM
EPOCHS     = 40
LR         = 3e-4
PATIENCE   = 7    # early stop on val_loss (not val_acc — more reliable)
WARMUP_EP  = 3    # linear LR warmup

# Blocks to unfreeze per modality
# X-Ray: unfreeze more (larger, more diverse dataset)
# CT/MRI: fewer blocks (smaller datasets — risk of overfit)
UNFREEZE_BLOCKS = {
    'xray': ('features.6', 'features.7', 'features.8'),
    'ct':   ('features.7', 'features.8'),
    'mri':  ('features.7', 'features.8'),
}

# ── Print system info ─────────────────────────────────────────────────────────
print(f'Device     : {DEVICE}')
print(f'AMP        : {USE_AMP}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Epochs     : {EPOCHS} (max, early stop patience={PATIENCE})')
if DEVICE.type == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    import psutil
    print(f'RAM        : {psutil.virtual_memory().available/1024**3:.1f} GB available')

print()
print('Verifying preprocessed splits...')
all_ok = True
for mod in ['xray', 'ct', 'mri']:
    for split in ['train', 'val', 'test']:
        for fname in ['X.npy', 'y.npy']:
            fp = DATA_DIR / mod / split / fname
            if not fp.exists():
                print(f'  ❌ MISSING: {fp}')
                all_ok = False
if all_ok:
    print('  ✅ All splits found. Ready to train.')
else:
    raise FileNotFoundError('Run preprocessing.ipynb first!')


Device     : cuda
AMP        : True
Batch size : 32
Epochs     : 40 (max, early stop patience=7)
GPU        : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM       : 4.0 GB

Verifying preprocessed splits...
  ✅ All splits found. Ready to train.


In [2]:
# ── CELL 2 — Dataset & DataLoader ────────────────────────────────────────────

IMAGENET_NORM = T.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])

def get_train_transform():
    """
    Augmentation policy for training.
    Conservative — preserves medically relevant features.
    No vertical flip (X-rays are never upside down).
    No heavy color jitter (grayscale modalities use stacked channels).
    """
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=12),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.05),
        T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        T.ToTensor(),
        IMAGENET_NORM,
    ])

def get_eval_transform():
    return T.Compose([T.ToTensor(), IMAGENET_NORM])


class MedicalImageDataset(Dataset):
    """
    Memory-mapped dataset.
    Only one sample loaded per __getitem__ call.
    Supports both X.npy (float32) and falls back gracefully.
    """
    def __init__(self, modality: str, split: str, transform=None):
        base   = DATA_DIR / modality / split
        x_path = str(base / 'X.npy')
        y_path = str(base / 'y.npy')

        self.y = np.load(y_path).astype(np.int64)
        n      = len(self.y)

        # Validate memmap size strictly
        raw = np.memmap(x_path, dtype='float32', mode='r')
        expected = n * IMG_SIZE * IMG_SIZE * 3
        if raw.size != expected:
            raise ValueError(
                f'Shape mismatch in {x_path}\n'
                f'  Got: {raw.size} elements\n'
                f'  Expected: {expected} ({n} × {IMG_SIZE} × {IMG_SIZE} × 3)\n'
                f'  Re-run preprocessing.ipynb'
            )
        del raw

        self.X         = np.memmap(x_path, dtype='float32', mode='r',
                                   shape=(n, IMG_SIZE, IMG_SIZE, 3))
        self.transform = transform
        self.modality  = modality
        self.split     = split

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img   = np.array(self.X[idx], copy=True)   # float32 HWC [0,1]
        label = int(self.y[idx])

        # Validate pixel range (catches preprocessing errors at load time)
        if img.max() > 1.01 or img.min() < -0.01:
            img = np.clip(img, 0.0, 1.0)

        if self.transform:
            img_uint8 = np.clip(img * 255, 0, 255).astype(np.uint8)
            img = self.transform(PILImage.fromarray(img_uint8))
        else:
            img = torch.from_numpy(
                np.ascontiguousarray(img)).permute(2, 0, 1).float()
            img = IMAGENET_NORM(img)
        return img, label


def make_loaders(modality: str):
    train_ds = MedicalImageDataset(modality, 'train', get_train_transform())
    val_ds   = MedicalImageDataset(modality, 'val',   get_eval_transform())
    test_ds  = MedicalImageDataset(modality, 'test',  get_eval_transform())

    kw = dict(
        batch_size         = BATCH_SIZE,
        num_workers        = 0,      # 0 = single process — safe everywhere
        pin_memory         = PIN_MEMORY,
        persistent_workers = False,
    )
    classes = np.load(DATA_DIR / modality / 'classes.npy', allow_pickle=True)
    print(f'  [{modality}] Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')
    print(f'           Classes ({len(classes)}): {list(classes)}')
    return (DataLoader(train_ds, shuffle=True,  **kw),
            DataLoader(val_ds,   shuffle=False, **kw),
            DataLoader(test_ds,  shuffle=False, **kw),
            classes)


# ── MixUp ─────────────────────────────────────────────────────────────────────
def mixup_batch(X, y, alpha=0.3):
    """
    MixUp regularisation: interpolate pairs of images and labels.
    lam >= 0.5 always → dominant label preserved.
    α=0.3 is conservative — won't blur anatomical features too much.
    """
    if alpha <= 0:
        return X, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    lam = max(lam, 1.0 - lam)   # always >= 0.5
    idx = torch.randperm(X.size(0), device=X.device)
    return lam * X + (1.0 - lam) * X[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)

print('✅ Dataset & loaders ready.')


✅ Dataset & loaders ready.


In [3]:
# ── CELL 3 — ROIClassifier (EfficientNet-B3 + GradCAM++) ─────────────────────

class ROIClassifier(nn.Module):
    """
    EfficientNet-B3 backbone + custom classification head.

    Design:
    - Backbone pretrained on ImageNet (transfer learning)
    - Only unfreeze_blocks are trainable → prevents catastrophic forgetting
    - Head: Dropout(0.5) → Linear(256) → ReLU → Dropout(0.3) → Linear(n_classes)
    - GradCAM++ hooks on last feature block for ROI localisation

    Why EfficientNet-B3?
    - 12M parameters (much smaller than ResNet-50 at 25M)
    - Compound scaling → better feature extraction at 224×224
    - Widely validated on medical imaging benchmarks
    """
    def __init__(self, num_classes: int, dropout: float = 0.5,
                 unfreeze_blocks=None):
        super().__init__()
        if unfreeze_blocks is None:
            unfreeze_blocks = ('features.6', 'features.7', 'features.8')

        base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

        # Step 1: Freeze ALL backbone parameters
        for p in base.parameters():
            p.requires_grad = False

        # Step 2: Unfreeze only specified blocks
        for name, p in base.named_parameters():
            if any(blk in name for blk in unfreeze_blocks):
                p.requires_grad = True

        self.features = base.features
        self.avgpool  = base.avgpool
        in_feat       = base.classifier[1].in_features  # 1536 for B3

        # Head with two dropout stages
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),            # 0.5 — primary regularisation
            nn.Linear(in_feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.6),      # 0.3 — secondary regularisation
            nn.Linear(256, num_classes),
        )

        # GradCAM++ hooks on the last feature block
        self._grads = None
        self._acts  = None
        self.features[-1].register_forward_hook(self._hook_act)
        self.features[-1].register_full_backward_hook(self._hook_grad)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f'  Trainable : {trainable:,} / {total:,} '
              f'({100*trainable/total:.1f}%)')
        print(f'  Blocks    : {unfreeze_blocks}')

    def _hook_act(self, m, inp, out):
        self._acts  = out.detach()

    def _hook_grad(self, m, g_in, g_out):
        self._grads = g_out[0].detach()

    def forward(self, x):
        feat   = self.features(x)
        pooled = self.avgpool(feat).flatten(1)
        return self.classifier(pooled)

    def gradcam_plus_plus(self, img_tensor: torch.Tensor,
                           class_idx: int = None):
        """
        GradCAM++ inference.
        Works on both CPU and CUDA (no torch.no_grad() — backward needs grads).
        Returns:
            cam       : np.ndarray [224,224] float32 in [0,1]
            class_idx : int
            probs     : np.ndarray [num_classes] float32
        """
        self.eval()
        device = next(self.parameters()).device
        inp    = img_tensor.unsqueeze(0).to(device)

        with torch.enable_grad():
            inp    = inp.requires_grad_(True)
            output = self(inp)
            if class_idx is None:
                class_idx = int(output.argmax(dim=1).item())
            self.zero_grad()
            output[0, class_idx].backward()

        grads = self._grads[0]   # (C, H, W)
        acts  = self._acts[0]    # (C, H, W)

        # GradCAM++ weighting (better than GradCAM for small lesion localisation)
        gsq   = grads ** 2
        gcb   = grads ** 3
        denom = 2.0 * gsq + (acts * gcb).sum(dim=(1, 2), keepdim=True)
        denom = torch.where(denom != 0, denom, torch.ones_like(denom))
        alpha   = gsq / denom
        weights = (alpha * torch.relu(grads)).sum(dim=(1, 2))
        cam     = (weights[:, None, None] * acts).sum(0)
        cam     = torch.relu(cam).cpu().numpy()

        # Post-process: resize → smooth → threshold
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam.astype(np.float32), (IMG_SIZE, IMG_SIZE))
        cam = cv2.GaussianBlur(cam, (11, 11), 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        # 30% threshold removes diffuse background noise
        cam = np.where(cam < 0.30, 0.0, cam).astype(np.float32)
        if cam.max() > 0:
            cam = cam / cam.max()

        probs = torch.softmax(output, dim=1)[0].detach().cpu().numpy()

        if device.type == 'cuda':
            torch.cuda.empty_cache()

        return cam, class_idx, probs


print('✅ ROIClassifier defined.')


✅ ROIClassifier defined.


In [4]:
# ── CELL 4 — Training Functions ──────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    """One training epoch with MixUp and optional AMP."""
    model.train()
    total_loss = correct = total = 0

    for X, y in loader:
        X = X.float().to(DEVICE, non_blocking=PIN_MEMORY)
        y = y.to(DEVICE, non_blocking=PIN_MEMORY)

        X, y_a, y_b, lam = mixup_batch(X, y, alpha=0.3)
        optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with torch.amp.autocast('cuda'):
                out  = model(X)
                loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(X)
            loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y_a).sum().item()
        total      += X.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluation on val or test set."""
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []

    for X, y in loader:
        X = X.float().to(DEVICE, non_blocking=PIN_MEMORY)
        y = y.to(DEVICE, non_blocking=PIN_MEMORY)

        if USE_AMP:
            with torch.amp.autocast('cuda'):
                out  = model(X)
                loss = criterion(out, y)
        else:
            out  = model(X)
            loss = criterion(out, y)

        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += X.size(0)
        all_probs.append(torch.softmax(out, 1).cpu().numpy())
        all_labels.append(y.cpu().numpy())

    return (total_loss / total,
            correct / total,
            np.concatenate(all_probs),
            np.concatenate(all_labels))


def warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    """Linear LR warmup: prevents unstable large updates in early epochs."""
    if epoch <= warmup_epochs:
        for pg in optimizer.param_groups:
            pg['lr'] = base_lr * (epoch / warmup_epochs)


print('✅ Training functions ready.')
print()
print('Anti-overfitting checklist:')
print('  ✅ MixUp α=0.3           → regularises by interpolating samples')
print('  ✅ Label smoothing 0.15  → prevents overconfident softmax outputs')
print('  ✅ Dropout 0.5/0.3       → two-stage head regularisation')
print('  ✅ Weight decay 2e-4     → L2 penalty on all weights')
print('  ✅ Early stop (patience=7) → stops at generalisation peak')
print('  ✅ Grad clipping (norm=1) → prevents exploding gradients')


✅ Training functions ready.

Anti-overfitting checklist:
  ✅ MixUp α=0.3           → regularises by interpolating samples
  ✅ Label smoothing 0.15  → prevents overconfident softmax outputs
  ✅ Dropout 0.5/0.3       → two-stage head regularisation
  ✅ Weight decay 2e-4     → L2 penalty on all weights
  ✅ Early stop (patience=7) → stops at generalisation peak
  ✅ Grad clipping (norm=1) → prevents exploding gradients


In [5]:
# ── CELL 5 — train_modality() ────────────────────────────────────────────────

def train_modality(modality: str, epochs: int = EPOCHS, lr: float = LR):
    print(f'\n' + '='*65)
    print(f'  Training: {modality.upper()}')
    print('='*65)

    train_loader, val_loader, test_loader, classes = make_loaders(modality)
    num_classes = len(classes)

    if num_classes < 2:
        print(f'  ⚠️  SKIPPING — only {num_classes} class. Need ≥2 for classification.')
        return None, None, classes, None, None, test_loader

    model = ROIClassifier(
        num_classes,
        dropout         = 0.5,
        unfreeze_blocks = UNFREEZE_BLOCKS[modality]
    ).to(DEVICE)

    # Label smoothing 0.15 → model can never learn to output 100% confidence
    criterion = nn.CrossEntropyLoss(label_smoothing=0.15)

    # AdamW: Adam with proper weight decay (not L2 regularisation bug in Adam)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=2e-4
    )

    # CosineAnnealing: LR smoothly decays to near-zero → avoids sharp drop
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - WARMUP_EP, 1), eta_min=1e-6
    )

    # GradScaler ONLY on CUDA (raises error on CPU otherwise)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    best_val_loss  = float('inf')
    best_val_acc   = 0.0
    patience_count = 0
    best_path      = MODEL_DIR / f'{modality}_best.pth'

    print(f'  Criterion    : CrossEntropyLoss(label_smoothing=0.15)')
    print(f'  Optimizer    : AdamW(lr={lr}, weight_decay=2e-4)')
    print(f'  Scheduler    : CosineAnnealingLR + {WARMUP_EP}-ep warmup')
    print(f'  Early stop   : val_loss patience={PATIENCE}')
    print()

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # LR warmup overrides scheduler for first WARMUP_EP epochs
        if epoch <= WARMUP_EP:
            warmup_lr(optimizer, epoch, WARMUP_EP, lr)

        tr_loss, tr_acc       = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)

        if epoch > WARMUP_EP:
            scheduler.step()

        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        improved = vl_loss < best_val_loss
        if improved:
            best_val_loss  = vl_loss
            best_val_acc   = vl_acc
            patience_count = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'classes':          list(classes),
                'num_classes':      num_classes,
                'modality':         modality,
                'img_size':         IMG_SIZE,
                'best_val_loss':    best_val_loss,
                'best_val_acc':     best_val_acc,
                'imagenet_mean':    [0.485, 0.456, 0.406],
                'imagenet_std':     [0.229, 0.224, 0.225],
                'unfreeze_blocks':  UNFREEZE_BLOCKS[modality],
                'dropout':          0.5,
            }, best_path)
        else:
            patience_count += 1

        gap     = (tr_acc - vl_acc) * 100
        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        marker  = ' ✓ saved' if improved else f' (wait {patience_count}/{PATIENCE})'

        # Overfitting warning inline
        warn = ''
        if gap > 10: warn = ' ⚠️ overfit'
        elif gap > 5: warn = ' 🟡 mild gap'

        print(f'  Ep {epoch:3d}/{epochs} | '
              f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% '
              f'[gap {gap:+.1f}%{warn}] | '
              f'LR {lr_now:.2e} | '
              f'{elapsed:.0f}s{marker}')

        if patience_count >= PATIENCE:
            print(f'  ⏹  Early stop at epoch {epoch} (val_loss no improvement for {PATIENCE} epochs)')
            break

    # ── Load best checkpoint and evaluate on test set ─────────────────────────
    print(f'\n  Loading best checkpoint from epoch '
          f'{int(np.argmin(history["val_loss"]))+1}...')
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion)

    final_gap = (history['train_acc'][-1] - history['val_acc'][-1]) * 100
    best_ep   = int(np.argmin(history['val_loss'])) + 1
    health    = ('✅ well-generalised' if final_gap <= 5
                 else '🟡 mild overfit'   if final_gap <= 10
                 else '⚠️  overfitting — consider reducing unfreeze_blocks')

    print(f'\n  ✅ Best val_loss       : {best_val_loss:.4f}')
    print(f'     Best val_acc        : {best_val_acc*100:.2f}%')
    print(f'     Test accuracy       : {test_acc*100:.2f}%')
    print(f'     Train/val gap       : {final_gap:+.1f}%  {health}')
    print(f'     Best epoch          : {best_ep}/{len(history["val_loss"])}')
    print(f'     Saved to            : {best_path}')

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return model, history, classes, test_probs, test_labels, test_loader


print('✅ train_modality() ready.')
print()
print('Expected accuracy ranges (healthy generalisation):')
print('  X-Ray  : 85–93% test acc  (4-class, 21k images)')
print('  CT Scan: 80–90% test acc  (4-class, smaller dataset)')
print('  MRI    : 85–93% test acc  (4-class, Kaggle dataset)')
print()
print('  >95% likely = overfitting. Early stopping should prevent this.')
print('  <75% likely = underfitting. Try unfreezing more blocks.')


✅ train_modality() ready.

Expected accuracy ranges (healthy generalisation):
  X-Ray  : 85–93% test acc  (4-class, 21k images)
  CT Scan: 80–90% test acc  (4-class, smaller dataset)
  MRI    : 85–93% test acc  (4-class, Kaggle dataset)

  >95% likely = overfitting. Early stopping should prevent this.
  <75% likely = underfitting. Try unfreezing more blocks.


In [6]:
# ── CELL 6 — Train All Modalities ────────────────────────────────────────────
# This cell trains all three modalities sequentially.
# On GPU: ~15–30 min per modality depending on dataset size.
# On CPU: ~2–4 hours per modality (feasible but slow).
#
# To train only one modality, comment out the others below.

results = {}

for modality in ['xray', 'ct', 'mri']:
    model, history, classes, test_probs, test_labels, test_loader = \
        train_modality(modality)
    results[modality] = {
        'model':       model,
        'history':     history,
        'classes':     list(classes),
        'test_probs':  test_probs,
        'test_labels': test_labels,
        'test_loader': test_loader,
        'trained':     model is not None,
    }
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

print('\n' + '🎉 '*20)
print('Training complete!')
print()
for mod, r in results.items():
    if r['trained'] and r['test_probs'] is not None:
        test_acc = float((np.argmax(r['test_probs'],1) == r['test_labels']).mean())
        print(f'  {mod.upper():6s}: ✅ Test Acc = {test_acc*100:.2f}%')
    else:
        print(f'  {mod.upper():6s}: ⚠️  Skipped or failed')



  Training: XRAY
  [xray] Train=14821, Val=3172, Test=3172
           Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
  Trainable : 8,900,578 / 11,090,732 (80.3%)
  Blocks    : ('features.6', 'features.7', 'features.8')
  Criterion    : CrossEntropyLoss(label_smoothing=0.15)
  Optimizer    : AdamW(lr=0.0003, weight_decay=2e-4)
  Scheduler    : CosineAnnealingLR + 3-ep warmup
  Early stop   : val_loss patience=7

  Ep   1/40 | Loss 0.9564/0.7160 | Acc 74.5%/87.9% [gap -13.4%] | LR 1.00e-04 | 242s ✓ saved
  Ep   2/40 | Loss 0.8463/0.6783 | Acc 83.4%/91.0% [gap -7.5%] | LR 2.00e-04 | 275s ✓ saved
  Ep   3/40 | Loss 0.8119/0.6415 | Acc 85.8%/92.2% [gap -6.3%] | LR 3.00e-04 | 246s ✓ saved
  Ep   4/40 | Loss 0.7931/0.6302 | Acc 87.1%/92.9% [gap -5.8%] | LR 2.99e-04 | 305s ✓ saved
  Ep   5/40 | Loss 0.7868/0.6133 | Acc 87.7%/93.1% [gap -5.5%] | LR 2.98e-04 | 269s ✓ saved
  Ep   6/40 | Loss 0.7617/0.6396 | Acc 88.8%/93.6% [gap -4.8%] | LR 2.95e-04 | 256s (wait 1/7)
  Ep   

In [7]:
# ── CELL 7 — Training Curves ─────────────────────────────────────────────────
trained_mods = [m for m in ['xray', 'ct', 'mri'] if results[m]['trained']]
n = len(trained_mods)

if n == 0:
    print('No modalities trained.')
else:
    ncols = n
    fig, axes = plt.subplots(2, ncols, figsize=(7 * ncols, 10))
    if ncols == 1:
        axes = axes[:, np.newaxis]
    fig.suptitle(
        'Training Curves\n'
        'MixUp α=0.3  |  Label Smoothing 0.15  |  Early Stop on val_loss',
        fontsize=13, fontweight='bold')

    for col, modality in enumerate(trained_mods):
        h    = results[modality]['history']
        eps  = range(1, len(h['train_loss']) + 1)
        best = int(np.argmin(h['val_loss'])) + 1

        ax0 = axes[0, col]
        ax0.plot(eps, h['train_loss'], label='Train', color='royalblue', lw=2)
        ax0.plot(eps, h['val_loss'],   label='Val',   color='tomato',    lw=2)
        ax0.axvline(best, color='green', ls='--', lw=1.5, label=f'Best ep {best}')
        ax0.set_title(f'{modality.upper()} — Loss', fontweight='bold')
        ax0.legend(); ax0.set_xlabel('Epoch')
        ax0.spines[['top', 'right']].set_visible(False)

        ax1 = axes[1, col]
        tr_pct = [a * 100 for a in h['train_acc']]
        vl_pct = [a * 100 for a in h['val_acc']]
        ax1.plot(eps, tr_pct, label='Train', color='royalblue', lw=2)
        ax1.plot(eps, vl_pct, label='Val',   color='tomato',    lw=2)
        ax1.fill_between(eps, vl_pct, tr_pct,
                         alpha=0.12, color='red', label='Gap (overfit signal)')
        ax1.axvline(best, color='green', ls='--', lw=1.5)
        ax1.axhline(95, color='orange', ls=':', lw=1, label='95% warning')
        ax1.set_title(f'{modality.upper()} — Accuracy (%)', fontweight='bold')
        ax1.legend(fontsize=8); ax1.set_xlabel('Epoch')
        ax1.set_ylim([0, 100])
        ax1.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    out_path = 'models/training_curves.png'
    plt.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out_path}')
    print()
    print('Interpretation guide:')
    print('  Blue line above red line  → train/val gap (smaller = better generalisation)')
    print('  Orange dashed at 95%      → anything above may indicate overfitting')
    print('  Green vertical line       → best epoch (where checkpoint was saved)')


✅ Saved → models/training_curves.png

Interpretation guide:
  Blue line above red line  → train/val gap (smaller = better generalisation)
  Orange dashed at 95%      → anything above may indicate overfitting
  Green vertical line       → best epoch (where checkpoint was saved)


In [8]:
# ── CELL 8 — Evaluation: Classification Report + Confusion Matrix + ROC ──────

def plot_confusion_matrix(labels, preds, classes, modality):
    disp = ConfusionMatrixDisplay(
        confusion_matrix(labels, preds), display_labels=classes)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{modality.upper()} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    path = f'models/{modality}_confusion_matrix.png'
    plt.savefig(path, dpi=120)
    plt.show()
    print(f'  Saved → {path}')


def plot_roc_curves(labels, probs, classes, modality):
    n_cls = len(classes)
    y_bin = label_binarize(labels, classes=range(n_cls))
    colors = plt.cm.Set1(np.linspace(0, 0.8, n_cls))
    auc_scores = []
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, (cls, color) in enumerate(zip(classes, colors)):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            auc         = roc_auc_score(y_bin[:, i], probs[:, i])
            auc_scores.append(auc)
            ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC={auc:.3f})')
        except Exception as e:
            print(f'  ROC skip {cls}: {e}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{modality.upper()} — ROC Curves', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    path = f'models/{modality}_roc_curves.png'
    plt.savefig(path, dpi=120)
    plt.show()
    print(f'  Saved → {path}')
    return auc_scores


for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        print(f'\n{modality.upper()} — Skipped.'); continue

    preds = np.argmax(r['test_probs'], axis=1)
    print(f'\n' + '='*55)
    print(f'  {modality.upper()} — Test Set Classification Report')
    print('='*55)
    print(classification_report(
        r['test_labels'], preds, target_names=r['classes']))

    plot_confusion_matrix(r['test_labels'], preds, r['classes'], modality)
    auc = plot_roc_curves(r['test_labels'], r['test_probs'], r['classes'], modality)
    if auc:
        print(f'  Mean AUC: {np.mean(auc):.4f}')



  XRAY — Test Set Classification Report
                 precision    recall  f1-score   support

          COVID       0.98      0.96      0.97       542
   Lung_Opacity       0.96      0.90      0.93       901
         Normal       0.93      0.97      0.95      1528
Viral Pneumonia       0.97      0.97      0.97       201

       accuracy                           0.95      3172
      macro avg       0.96      0.95      0.95      3172
   weighted avg       0.95      0.95      0.95      3172

  Saved → models/xray_confusion_matrix.png
  Saved → models/xray_roc_curves.png
  Mean AUC: 0.9908

  CT — Test Set Classification Report
                         precision    recall  f1-score   support

         adenocarcinoma       0.89      0.47      0.62       120
   large cell carcinoma       0.52      0.88      0.66        51
                 normal       0.88      0.98      0.93        54
squamous cell carcinoma       0.73      0.86      0.79        90

               accuracy            

In [9]:
# ── CELL 9 — GradCAM++ ROI Visualisation ─────────────────────────────────────
# Visualises the Region of Interest heatmaps on test set samples.
# This is the core output of the project: explainable ROI localisation.

IMAGENET_MEAN_VIS = np.array([0.485, 0.456, 0.406])
IMAGENET_STD_VIS  = np.array([0.229, 0.224, 0.225])


def denormalize(tensor_img: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalisation for display."""
    img = tensor_img.cpu().permute(1, 2, 0).numpy()
    return np.clip(img * IMAGENET_STD_VIS + IMAGENET_MEAN_VIS, 0, 1)


def overlay_cam(orig_float: np.ndarray, cam: np.ndarray,
                alpha: float = 0.5) -> np.ndarray:
    """Blend GradCAM++ heatmap onto original image."""
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mask = cam[:, :, np.newaxis]
    return np.clip(alpha * mask * heat + (1.0 - alpha * mask) * orig_float, 0, 1)


def visualize_roi(modality: str, n_samples: int = 4):
    r       = results[modality]
    classes = r['classes']

    if r['trained'] and r['model'] is not None:
        model = r['model']
    else:
        ckpt_path = MODEL_DIR / f'{modality}_best.pth'
        if ckpt_path.exists():
            ckpt  = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            model = ROIClassifier(
                ckpt['num_classes'],
                unfreeze_blocks=UNFREEZE_BLOCKS[modality]
            ).to(DEVICE)
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            print(f'  ⚠️  {modality}: no checkpoint found.')
            return

    print(f'\n🔍 {modality.upper()} — GradCAM++ ROI Localisation')

    X_batch, y_batch = next(iter(r['test_loader']))
    n_samples = min(n_samples, len(X_batch))
    X_batch   = X_batch[:n_samples].float()

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(
        f'{modality.upper()} — GradCAM++ ROI Localisation\n'
        f'Left: Original  |  Middle: GradCAM++ Heatmap  |  Right: Overlay',
        fontsize=12, fontweight='bold')

    for i in range(n_samples):
        img_t    = X_batch[i]
        true_lbl = int(y_batch[i].item())
        cam, pred_cls, probs = model.gradcam_plus_plus(img_t)
        orig = denormalize(img_t)
        over = overlay_cam(orig, cam)

        true_name = classes[true_lbl] if true_lbl < len(classes) else str(true_lbl)
        pred_name = classes[pred_cls] if pred_cls < len(classes) else str(pred_cls)
        status    = '✅' if true_lbl == pred_cls else '❌'

        axes[i, 0].imshow(orig); axes[i, 0].axis('off')
        axes[i, 0].set_title(f'Original | True: {true_name}', fontsize=9)

        axes[i, 1].imshow(cam, cmap='jet', vmin=0, vmax=1); axes[i, 1].axis('off')
        axes[i, 1].set_title(f'GradCAM++ | Conf: {probs[pred_cls]:.1%}', fontsize=9)

        axes[i, 2].imshow(over); axes[i, 2].axis('off')
        axes[i, 2].set_title(f'{status} Pred: {pred_name}', fontsize=9)

    plt.tight_layout()
    out = f'models/{modality}_roi_gradcam.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Saved → {out}')


for modality in ['xray', 'ct', 'mri']:
    if results[modality]['trained']:
        visualize_roi(modality)



🔍 XRAY — GradCAM++ ROI Localisation
  ✅ Saved → models/xray_roi_gradcam.png

🔍 CT — GradCAM++ ROI Localisation
  ✅ Saved → models/ct_roi_gradcam.png

🔍 MRI — GradCAM++ ROI Localisation
  ✅ Saved → models/mri_roi_gradcam.png


In [10]:
# ── CELL 10 — Final Summary & Per-Class Report ────────────────────────────────
import json as _json

summary = {}
for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        summary[modality] = {'test_accuracy': None, 'note': 'Skipped.'}
        continue

    preds = np.argmax(r['test_probs'], axis=1)
    acc   = float((preds == r['test_labels']).mean())
    prec, rec, f1, sup = precision_recall_fscore_support(
        r['test_labels'], preds, zero_division=0)

    summary[modality] = {
        'test_accuracy': acc,
        'classes': list(r['classes']),
        'per_class': {
            cls: {
                'precision': float(prec[i]),
                'recall':    float(rec[i]),
                'f1':        float(f1[i]),
                'support':   int(sup[i]),
            }
            for i, cls in enumerate(r['classes'])
        }
    }

with open(MODEL_DIR / 'summary.json', 'w') as f:
    _json.dump(summary, f, indent=2)

print('=' * 65)
print('FINAL MODEL SUMMARY')
print('=' * 65)
for mod, info in summary.items():
    if info['test_accuracy'] is not None:
        print(f'  {mod.upper():6s}: Test Acc = {info["test_accuracy"]*100:.2f}%  '
              f'| Classes = {info.get("classes", [])}')
    else:
        print(f'  {mod.upper():6s}: {info.get("note", "skipped")}')

print()
print('PER-CLASS PERFORMANCE')
print(f'{"Modality":<8} {"Class":<34} {"Prec":>6} {"Rec":>6} {"F1":>6} {"N":>7}')
print('-' * 68)
for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        print(f'{modality.upper():<8} — skipped'); print(); continue
    preds = np.argmax(r['test_probs'], axis=1)
    prec, rec, f1, sup = precision_recall_fscore_support(
        r['test_labels'], preds, zero_division=0)
    for i, cls in enumerate(r['classes']):
        print(f'{modality.upper():<8} {cls:<34} '
              f'{prec[i]:>6.3f} {rec[i]:>6.3f} {f1[i]:>6.3f} {int(sup[i]):>7d}')
    print()

print('\n✅ All outputs saved to models/')
print()
print('Files ready for app.py:')
for fp in sorted(Path('models').glob('*.pth')):
    print(f'  {fp.name}  ({fp.stat().st_size / 1024**2:.1f} MB)')
print()
print('Run: streamlit run app.py')


FINAL MODEL SUMMARY
  XRAY  : Test Acc = 94.80%  | Classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
  CT    : Test Acc = 73.65%  | Classes = ['adenocarcinoma', 'large cell carcinoma', 'normal', 'squamous cell carcinoma']
  MRI   : Test Acc = 93.75%  | Classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

PER-CLASS PERFORMANCE
Modality Class                                Prec    Rec     F1       N
--------------------------------------------------------------------
XRAY     COVID                               0.978  0.963  0.970     542
XRAY     Lung_Opacity                        0.955  0.898  0.926     901
XRAY     Normal                              0.931  0.970  0.950    1528
XRAY     Viral Pneumonia                     0.970  0.965  0.968     201

CT       adenocarcinoma                      0.891  0.475  0.620     120
CT       large cell carcinoma                0.523  0.882  0.657      51
CT       normal                              0.883  0.981  0.930     